# 2_models/02 — Within- vs pan-stratum models

Fits a model *within* each cancer type (and each first-line treatment class) and compares it to a
**size-matched** pan-stratum comparator on that stratum's held-out patients. Size matching is the
point: an unmatched pan arm would win on training-set size alone.

| Script | Stratum | Output dir | Metrics file |
|---|---|---|---|
| `within_vs_pan_cancer_models` | `CANCER_TYPE` | `results/pan_vs_within_cancer/` | `metrics_by_cancer_type.csv` |
| `within_treatment_vs_pan_treatment_models` | first-line `TREATMENT_CLASSIFICATION` | `results/pan_vs_within_treatment/` | `metrics_by_treatment.csv` |

**Runs after** `1_data/03` (both fit from the `icd3_post` embedding prediction dataset + non-text
covariates) and **before** `4_figures/02` (`figures.prep.figure2` reads both metrics files). Does *not* depend
on the SLURM arrays or on `2_models/04`.

Strata floors are module constants in each script — `MIN_STRATUM_N = 500`, `MIN_TRAIN_N = 100`,
`MIN_HELDOUT_N = 30`. Change them there, not here. Both scripts checkpoint per stratum under
`<outdir>/checkpoints/` and fingerprint their inputs, so an interrupted run is simply re-executed
and a cohort rebuild upstream invalidates stale checkpoints on its own.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config  # noqa: E402


def check_inputs(preconditions: list[tuple[str, str]]) -> list[str]:
    """Report presence of each (label, path). Returns the missing labels; never raises."""
    missing = []
    for label, path in preconditions:
        ok = os.path.exists(path)
        if not ok:
            missing.append(label)
        print(f"[{'ok ' if ok else 'MISSING'}] {label:<22} {path}")
    print(f"\n{'All inputs present.' if not missing else str(len(missing)) + ' missing: ' + ', '.join(missing)}")
    return missing


def report_outputs(outputs: list[tuple[str, str]]) -> None:
    """Print size and mtime for each (label, path) that exists."""
    for label, path in outputs:
        if os.path.exists(path):
            mb = os.path.getsize(path) / 1e6
            mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(path)))
            print(f"[ok     ] {label:<26} {mb:>8.1f} MB   {mtime}")
        else:
            print(f"[missing] {label:<26} {path}")


def run_module(module: str, args: list[str] | None = None, env: dict | None = None,
               capture: bool = False) -> dict:
    """Run `python -m module` from REPO_ROOT. Returns {returncode, wall_s, stdout}."""
    cmd = [sys.executable, "-m", module, *(args or [])]
    started = time.perf_counter()
    run_env = {**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})}
    kwargs = dict(cwd=str(REPO_ROOT), env=run_env)
    if capture:
        kwargs.update(text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    proc = subprocess.run(cmd, **kwargs)
    return {"returncode": proc.returncode, "wall_s": time.perf_counter() - started,
            "stdout": (proc.stdout or "") if capture else ""}


print(f"repo root: {REPO_ROOT}")
print(f"Python:  {sys.executable}")

## Configuration

In [ ]:
import schemes

SCHEME = "icd3_post"   # what both scripts load; mirrored here for the precondition check

RUNS = [
    ("cancer",    "pipelines.trajectories.within_vs_pan_cancer_models",
     os.path.join(config.RESULTS_PATH, "pan_vs_within_cancer"),
     "metrics_by_cancer_type.csv", "CANCER_TYPE"),
    ("treatment", "pipelines.trajectories.within_treatment_vs_pan_treatment_models",
     os.path.join(config.RESULTS_PATH, "pan_vs_within_treatment"),
     "metrics_by_treatment.csv", "TREATMENT"),
]

ENABLED = {"cancer": True, "treatment": True}
SKIP_IF_DONE = False   # both resume from checkpoints, so re-running an interrupted run is cheap

print(f"results path: {config.RESULTS_PATH}")
print(f"scheme:       {SCHEME}")
print(f"enabled:      {', '.join(k for k, v in ENABLED.items() if v) or 'none'}")

## Preconditions

Both scripts read the embeddings and cancer types; the treatment script needs the treatment table
too. This cell does not raise — it distinguishes "the run failed" from "the run never had its
inputs".

In [ ]:
check_inputs([
    (f"{SCHEME} embeddings", os.path.join(config.SURV_PATH, schemes.embedding_file(SCHEME))),
    ("cancer types",         os.path.join(config.FEATURE_PATH, "cancer_type_df.csv.gz")),
    ("treatment by line",    os.path.join(config.FEATURE_PATH,
                                          "categorical_treatment_data_by_line.csv.gz")),
])

## Run

One subprocess per comparison so no state leaks between them. Output streams straight through —
these are long runs behind per-stratum progress. A failure in one does not stop the other; neither
reads the other's output. Interrupting leaves completed strata checkpointed on disk.

In [ ]:
results = []

for label, module, outdir, fname, _stratum_col in RUNS:
    if not ENABLED[label]:
        print(f"\n=== {label}: disabled ===")
        results.append((label, "disabled", 0.0))
        continue
    if SKIP_IF_DONE and os.path.exists(os.path.join(outdir, fname)):
        print(f"\n=== {label}: already done ({fname} present) ===")
        results.append((label, "skipped", 0.0))
        continue

    print(f"\n{'=' * 78}\n=== {label}: python -m {module}\n{'=' * 78}", flush=True)
    outcome = run_module(module)
    status = "ok" if outcome["returncode"] == 0 else f"FAILED (exit {outcome['returncode']})"
    print(f"\n[{label}] {status} after {outcome['wall_s'] / 60:.1f} min")
    results.append((label, status, outcome["wall_s"]))

print(f"\n{'=' * 78}\n=== Run summary ===")
for label, status, elapsed in results:
    print(f"  {label:<10} {status}" + (f"  ({elapsed / 60:.1f} min)" if elapsed else ""))

## Results

Per-stratum held-out comparison. `DELTA_AUC_WITHIN_MINUS_PAN > 0` means the within-stratum model
beat its size-matched pan comparator on that stratum's held-out patients. The `Overall` row is the
whole held-out set, not a stratum. Every read is guarded, so this is safe on a partial pipeline.

In [ ]:
import polars as pl

METRIC_COLS = ["CINDEX_PAN", "CINDEX_WITHIN", "AUC_PAN", "AUC_WITHIN",
               "DELTA_AUC_WITHIN_MINUS_PAN", "DELTA_WITHIN_MINUS_PAN"]

with pl.Config(tbl_rows=60, tbl_cols=12, tbl_width_chars=160, float_precision=3):
    for label, _module, outdir, fname, stratum_col in RUNS:
        path = os.path.join(outdir, fname)
        print(f"\n{'=' * 78}\n=== {label}: {fname}\n{'=' * 78}")

        report_outputs([
            ("train_risk_scores.csv",    os.path.join(outdir, "train_risk_scores.csv")),
            ("held_out_risk_scores.csv", os.path.join(outdir, "held_out_risk_scores.csv")),
            (fname,                      path),
        ])
        ckpt = os.path.join(outdir, "checkpoints")
        n_ckpt = len([n for n in os.listdir(ckpt) if not n.startswith(".")]) if os.path.isdir(ckpt) else 0
        print(f"           {n_ckpt} checkpoint file(s) — strata completed\n")

        if not os.path.exists(path):
            print("  metrics not written — run has not completed")
            continue

        metrics = pl.read_csv(path)
        overall = metrics.filter(pl.col(stratum_col) == "Overall")
        strata = metrics.filter(pl.col(stratum_col) != "Overall")

        if overall.height:
            row = overall.row(0, named=True)
            print(f"Overall held-out (n={row['N_HELDOUT']:,}): "
                  f"AUC pan={row['AUC_PAN']:.3f} within={row['AUC_WITHIN']:.3f} "
                  f"(delta {row['DELTA_AUC_WITHIN_MINUS_PAN']:+.3f})   |   "
                  f"C-index pan={row['CINDEX_PAN']:.3f} within={row['CINDEX_WITHIN']:.3f} "
                  f"(delta {row['DELTA_WITHIN_MINUS_PAN']:+.3f})")

        n_wins = int((strata["DELTA_AUC_WITHIN_MINUS_PAN"] > 0).sum())
        print(f"{strata.height} strata compared; within-stratum wins on AUC in "
              f"{n_wins}/{strata.height}\n")
        print(metrics.select([stratum_col, "N_HELDOUT"]
                             + [c for c in METRIC_COLS if c in metrics.columns]))

## Hand-off to 4_figures/02

A missing metrics file means Figure 2's within-vs-pan panels will be absent or stale — `4_figures/02` will not
fail loudly.

In [ ]:
ready = all(os.path.exists(os.path.join(outdir, fname)) for _, _, outdir, fname, _ in RUNS)
check_inputs([(f"figure2 input ({label})", os.path.join(outdir, fname))
              for label, _, outdir, fname, _ in RUNS])

print(f"\nfigure data dir: {config.FIGURE_DATA_DIR}")
report_outputs([(csv, os.path.join(config.FIGURE_DATA_DIR, csv))
                for csv in ("fig2_within_vs_pan_cancer.csv", "fig2_within_vs_pan_treatment.csv")])

print("\n" + ("Ready for 4_figures/02." if ready else "Not ready — re-run the missing comparison above."))